# EXP002: Qwen3.5-2B on MolmoWeb
Thin Kaggle runner for pinned data preparation, untouched-model baseline, QLoRA SFT, post-SFT evaluation, and immutable result archiving. Enable Internet and a GPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_REV = "bb238b7"
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path("/kaggle/working/spider")
    if not (REPO_ROOT / "pyproject.toml").exists():
        subprocess.run(
            ["git", "clone", "https://github.com/yogesh-dhande/spider.git", str(REPO_ROOT)],
            check=True,
        )
if (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

In [ ]:
%pip install -q -r requirements/experiment2-kaggle.txt

In [ ]:
from spider.workflow import gpu_summary, restore_run

print(gpu_summary())
PREVIOUS_RUN_ROOT = None  # Example: '/kaggle/input/spider-exp002-run-1'
restore_run(PREVIOUS_RUN_ROOT, REPO_ROOT)

## Prepare the fixed domain-disjoint manifests

In [ ]:
from spider.prepare import prepare_all

prepare_all("configs/experiment2.yaml")

## Untouched Qwen3.5 baseline
For the first compatibility check, add `limit=8`; remove it for the official baseline. Predictions flush individually and resume safely.

In [ ]:
from spider.evaluate import evaluate

_, baseline_metrics = evaluate(
    "configs/experiment2.yaml", "baseline", None, ["molmoweb", "screenspot"]
)
baseline_metrics

## QLoRA SFT in resumable chunks

In [ ]:
from spider.train import train

train("configs/experiment2.yaml", additional_steps=500)

## Post-SFT evaluation, comparison, and publication archive

In [ ]:
from spider.archive import archive_results
from spider.workflow import compare_run_outputs

adapter = "outputs/experiment2/adapter/final"
evaluate("configs/experiment2.yaml", "sft", adapter, ["molmoweb", "screenspot"])
comparison_path, _ = compare_run_outputs("configs/experiment2.yaml")
print(comparison_path.read_text())
archive_results("configs/experiment2.yaml")